In [ ]:
# ============================================
# 0. Install required libraries
# ============================================
!pip install --upgrade transformers tqdm openpyxl

# ============================================
# 1. Configure the environment and import libraries
# ============================================
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

import os
import torch
import numpy as np
import pandas as pd
import transformers

from tqdm import tqdm
from datetime import datetime
from google.colab import drive
from transformers import AutoTokenizer, AutoModel

# ============================================
# 2. Mount Google Drive
# ============================================
drive.mount('/content/gdrive')

# ============================================
# 3. Configure paths and basic settings
# ============================================
# Path to the saved KLUE-RoBERTa-based MoE model (state_dict file saved during training)
MODEL_NAME = "klue/roberta-base"
weights_path = '/content/gdrive/MyDrive/10_Models/3-1_KLUE-MoE&OPN/best_model_weights.pt'  # Path to the saved model weights

# Configure the input and output folder paths
input_folder = '/content/gdrive/MyDrive/100_input/'   # Input file folder
output_folder = '/content/gdrive/MyDrive/101_output/'  # Output file folder

# Configure the log file path
log_file_path = os.path.join(output_folder, 'missing_files_log.txt')
os.makedirs(output_folder, exist_ok=True)  # Ensure that the folder exists (create it if necessary)

# Label mapping
label_mapping = {0: 'B', 1: 'N', 2: 'S'}
attribute_columns = [
    'INFOSRC', 'INFOSRC_CODE_2', 'ART_DATE', 'ART_PROVIDER', 'ART_CATEGORY1','ART_CATEGORY2', 'ART_CATEGORY3',
    'ART_TAG_1', 'ART_TAG_2', 'ART_TAG_3', 'SNT_TAG_1', 'SNT_TAG_2','SNT_TAG_3', 'ART_HEADLINE', 'ART_BYLINE'
]

# ============================================
# 4. Logging function
# ============================================
def write_log(message, input_file=None):
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    with open(log_file_path, 'a', encoding='utf-8') as log_file:
        if input_file:
            log_file.write(f"[{timestamp}] {message} - File: {input_file}\n")
        else:
            log_file.write(f"[{timestamp}] {message}\n")
    print(message if input_file is None else f"{message} - File: {input_file}")

with open(log_file_path, 'w', encoding='utf-8') as log_file:
    log_file.write(f"File processing log - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# ============================================
# 5. Define the MoE model class
# ============================================
class MoEBertModel(torch.nn.Module):
    def __init__(self, num_labels):
        super(MoEBertModel, self).__init__()
        self.num_labels = num_labels
        self.hidden_size = 768

        # Expert 1 (attribute-text input)
        self.bert_expert1 = AutoModel.from_pretrained(MODEL_NAME)

        # Expert 2 (quotation-text input)
        self.bert_expert2 = AutoModel.from_pretrained(MODEL_NAME)

        self.classifier1 = torch.nn.Linear(self.hidden_size, num_labels)
        self.classifier2 = torch.nn.Linear(self.hidden_size, num_labels)

        # Freeze some of the initial layers (e.g., the first six layers)
        for param in self.bert_expert1.encoder.layer[:6].parameters():
            param.requires_grad = False
        for param in self.bert_expert2.encoder.layer[:6].parameters():
            param.requires_grad = False

        # Gating network
        self.gating_network = torch.nn.Sequential(
            torch.nn.Linear(self.hidden_size * 2, self.hidden_size),
            torch.nn.GELU(),
            torch.nn.Dropout(0.1),
            torch.nn.Linear(self.hidden_size, 2),
            torch.nn.Softmax(dim=1)
        )

        self.dropout = torch.nn.Dropout(0.1)
        self.classifier = torch.nn.Linear(self.hidden_size, self.num_labels)

    def forward(self, input_ids1, attention_mask1, input_ids2, attention_mask2, label=None, return_gating=False):
        # Expert 1 output
        encoding1 = self.bert_expert1(input_ids=input_ids1, attention_mask=attention_mask1).pooler_output

        # Expert 2 output
        encoding2 = self.bert_expert2(input_ids=input_ids2, attention_mask=attention_mask2).pooler_output

        # Expert-specific logits
        logits1 = self.classifier1(encoding1)
        logits2 = self.classifier2(encoding2)

        # Gating weights
        combined = torch.cat((encoding1, encoding2), dim=1)
        gating_weights = self.gating_network(combined)

        # Combine the expert outputs
        expert_outputs = torch.stack([logits1, logits2], dim=1)
        gated_output = torch.einsum('bi,bih->bh', gating_weights, expert_outputs)

        if label is not None:
            criterion = torch.nn.CrossEntropyLoss()
            loss = criterion(gated_output, label)

            if return_gating:
                return loss, gated_output, gating_weights
            return loss, gated_output

        if return_gating:
            return gated_output, gating_weights

        return gated_output

# ============================================
# 6. Load the device, model, and tokenizer
# ============================================

# Configure the device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("weights exists:", os.path.exists(weights_path))
print("weights_path:", weights_path)

if not os.path.exists(weights_path):
    raise FileNotFoundError(f"Weights file not found: {weights_path}")

# Load the model
num_labels_from_mapping = len(label_mapping)
model = MoEBertModel(num_labels=num_labels_from_mapping)

# Load the state_dict
state_dict = torch.load(weights_path, map_location='cpu', weights_only=True)
missing_keys, unexpected_keys = model.load_state_dict(state_dict, strict=True)

print("missing_keys:", missing_keys)
print("unexpected_keys:", unexpected_keys)

model.to(device)
model.eval()

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

# ------------------------------
# 7. Data preprocessing
# ------------------------------
def build_attributes_text(df: pd.DataFrame) -> pd.DataFrame:
    existing_cols = [col for col in attribute_columns if col in df.columns]
    if len(existing_cols) > 0:
        df['attributes_text'] = df[existing_cols].fillna("").astype(str).agg(' '.join, axis=1)
    else:
        df['attributes_text'] = ""
    return df

def preprocess_data_bulk_pair(attr_texts, content_texts, max_len=256):
    try:
        tokenized_attr = tokenizer(
            attr_texts,
            max_length=max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='np'
        )

        tokenized_content = tokenizer(
            content_texts,
            max_length=max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='np'
        )

        return (
            np.array(tokenized_attr['input_ids']),
            np.array(tokenized_attr['attention_mask']),
            np.array(tokenized_content['input_ids']),
            np.array(tokenized_content['attention_mask'])
        )

    except Exception as e:
        write_log(f"An error occurred during preprocessing: {str(e)}")
        return np.array([]), np.array([]), np.array([]), np.array([])

# ============================================
# 8. Tone analysis function (inference)
# ============================================
def analyze_OPN(df, output_path, batch_size=16384, max_len=256): # Candidate batch sizes: 4,096, 8,192, and 16,384
    # Return if the 'STN_CONTENT' column is missing
    if 'STN_CONTENT' not in df.columns:
        write_log("Missing 'STN_CONTENT' column", input_file=output_path)
        return

    df = build_attributes_text(df)

    # Convert values to strings
    attr_texts = df['attributes_text'].astype(str).tolist()
    content_texts = df['STN_CONTENT'].astype(str).tolist()

    # Preprocess the data
    input_ids_attr, attention_masks_attr, input_ids_content, attention_masks_content = preprocess_data_bulk_pair(
        attr_texts, content_texts, max_len=max_len
    )

    if (input_ids_attr.size == 0 or attention_masks_attr.size == 0 or
        input_ids_content.size == 0 or attention_masks_content.size == 0):
        write_log("The preprocessed data are empty", input_file=output_path)
        return

    predictions_logits = []

    model.eval()
    with torch.no_grad():
        n = input_ids_attr.shape[0]

        for i in tqdm(range(0, n, batch_size), desc="Labeling progress"):
            batch_input_ids_attr = torch.tensor(input_ids_attr[i:i+batch_size], dtype=torch.long).to(device)
            batch_attention_masks_attr = torch.tensor(attention_masks_attr[i:i+batch_size], dtype=torch.long).to(device)
            batch_input_ids_content = torch.tensor(input_ids_content[i:i+batch_size], dtype=torch.long).to(device)
            batch_attention_masks_content = torch.tensor(attention_masks_content[i:i+batch_size], dtype=torch.long).to(device)

            try:
                with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                    outputs = model(batch_input_ids_attr, batch_attention_masks_attr,
                                    batch_input_ids_content, batch_attention_masks_content)
                predictions_logits.append(outputs.cpu().numpy())

            except Exception as e:
                write_log(f"An error occurred during prediction: {str(e)}", input_file=output_path)

    # Release GPU memory after processing all batches
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    if len(predictions_logits) == 0:
        write_log("No prediction results are available.", input_file=output_path)
        return

    # Concatenate the logits
    predictions_logits = np.concatenate(predictions_logits, axis=0)

    # Calculate softmax probabilities
    exp_logits = np.exp(predictions_logits - np.max(predictions_logits, axis=1, keepdims=True))
    probs = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)

    # Generate predicted labels
    pred_labels = np.argmax(predictions_logits, axis=1)

    # Save the results
    df['OPN'] = pred_labels
    df['OPN_Label'] = df['OPN'].map(label_mapping)
    df['Probability'] = np.max(probs, axis=1)

    try:
        df.to_excel(output_path, index=False)
        print(f"Results saved: {output_path}")
    except Exception as e:
        write_log(f"Failed to save the file: {str(e)}", input_file=output_path)

# ============================================
# 9. Main routine: run inference and save results for each file
# ============================================
input_files = [f for f in os.listdir(input_folder) if f.endswith('.xlsx')]

print(f"Total number of input files: {len(input_files)}")

for input_file in tqdm(input_files, desc="File processing progress"):
    input_path = os.path.join(input_folder, input_file)

    output_file_name = f"{os.path.splitext(input_file)[0]}_OPN.xlsx"
    output_path = os.path.join(output_folder, output_file_name)

    # Skip the file if its output file already exists
    if os.path.exists(output_path):
        write_log("Already processed", input_file=input_file)
        continue

    print(f"Processing: {input_file}")

    # Record missing input files in the log
    if not os.path.exists(input_path):
        write_log("File not found", input_file=input_file)
        continue

    try:
        usecols_target = set(['STN_CONTENT'] + attribute_columns)
        df = pd.read_excel(
            input_path,
            engine='openpyxl',
            usecols=lambda c: c in usecols_target
        )

        if 'STN_CONTENT' not in df.columns:
            write_log("Missing 'STN_CONTENT' column", input_file=input_file)
            continue

        analyze_OPN(df, output_path, batch_size=16384, max_len=256)  # Tone inference

    except Exception as e:
        write_log(f"Failed to process the file: {str(e)}", input_file=input_file)

# ============================================
# 10. Environment check
# ============================================
print("\n[Training environment check]")
print("PyTorch version:", torch.__version__)
print("Transformers version:", transformers.__version__)
print("Tokenizer class:", tokenizer.__class__.__name__)
print("Tokenizer:", tokenizer.name_or_path)
print("Device:", device)
print("All files have been processed.")